# OpenFloodAI: explore 20 dummy observations

**DEMONSTRATION ONLY. All values and reviews are invented. No image/mask files are included. Do not train on this.**

This illustrates a proposed visual-change analysis layer, not a claim that the app already exports these tables.
`observations.jsonl`: one row per image. `comparisons.jsonl`: one row per ordered pair.
The original images and masks would be additional files in a real dataset.

Run locally with Python, pandas and a Jupyter-compatible notebook viewer. Keep the JSONL files beside this notebook.
In Colab upload this notebook and the three JSON/JSONL files using the Files panel (or the loader below).
Uploading to Colab sends those files to Google; only this dummy data is intended for that demonstration.
No API calls, model training or external downloads are performed by these cells.

In [1]:
from pathlib import Path
import json
import pandas as pd

DATA = Path('.')
required = ['observations.jsonl', 'comparisons.jsonl', 'manifest.json']
if not all((DATA / name).exists() for name in required):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Run from the extracted dataset-demo folder, or set DATA to that folder.')
    print('Choose observations.jsonl, comparisons.jsonl and manifest.json from the dummy demo.')
    files.upload()
assert all((DATA / name).exists() for name in required), 'Required demo files missing'
manifest = json.loads((DATA / 'manifest.json').read_text())
assert manifest['is_demo'] and not manifest['training_ready']
observations = pd.read_json(DATA / 'observations.jsonl', lines=True)
comparisons = pd.read_json(DATA / 'comparisons.jsonl', lines=True)
print(f'{len(observations)} observations; {len(comparisons)} ordered pairs. DUMMY DATA ONLY.')
display(observations[['observation_id','captured_at_utc','scenario','mask_review',
                      'water_fraction_pct','matched_gauge_value','input_quality','split']])

20 observations; 15 ordered pairs. DUMMY DATA ONLY.


observation_id,captured_at_utc,scenario,mask_review,water_fraction_pct,matched_gauge_value,input_quality,split
A1,2025-06-01T10:00:00+00:00,Rising,accepted,30.0,3.20,clear,train
A2,2025-06-01T10:15:00+00:00,Rising,accepted,35.0,3.50,clear,train
A3,2025-06-01T10:30:00+00:00,Rising,accepted,42.0,4.10,clear,train
A4,2025-06-01T10:45:00+00:00,Rising,accepted,48.0,4.60,clear,train
B1,2025-06-08T10:00:00+00:00,Falling,accepted,48.0,4.60,clear,train
B2,2025-06-08T10:15:00+00:00,Falling,accepted,44.0,4.20,clear,train
B3,2025-06-08T10:30:00+00:00,Falling,accepted,39.0,3.80,clear,train
B4,2025-06-08T10:45:00+00:00,Falling,accepted,33.0,3.40,clear,train
C1,2025-06-15T10:00:00+00:00,High and steady,accepted,52.0,4.90,clear,validation
C2,2025-06-15T10:15:00+00:00,High and steady,accepted,52.0,4.91,clear,validation


## 1. Which examples have hypothetically accepted segmentation?
A moved-camera image can still have a correct water mask, but cannot be compared with the old reference.
Gauge matches are not required for segmentation. The `hypothetical` flag is not actual training eligibility:
all assets are absent here, so every row has `training_ready=False`.

In [2]:
accepted = observations[observations['hypothetical_segmentation_usable']]
display(accepted[['observation_id','mask_review','water_fraction_pct',
                  'gauge_match_status','hypothetical_change_usable']])
print('Hypothetical accepted masks:', len(accepted))
print('Actual training-ready examples:', observations['training_ready'].sum())

observation_id,mask_review,water_fraction_pct,gauge_match_status,hypothetical_change_usable
A1,accepted,30.0,matched,True
A2,accepted,35.0,matched,True
A3,accepted,42.0,matched,True
A4,accepted,48.0,matched,True
B1,accepted,48.0,matched,True
B2,accepted,44.0,matched,True
B3,accepted,39.0,matched,True
B4,accepted,33.0,matched,True
C1,accepted,52.0,matched,True
C2,accepted,52.0,matched,True


Hypothetical accepted masks: 18
Actual training-ready examples: 0


## 2. What changed, and what did a human say?
Human labels are independently invented examples, not generated by gauge thresholds.
The stable C sequence deliberately has small mask jitter despite a no-change label.
Percentage points/hour is image coverage change, not vertical velocity or river speed.

In [3]:
usable = comparisons[comparisons['comparison_status'] == 'usable']
display(usable[['pair_id','coverage_delta_pp','human_visual_label','matched_gauge_delta_ft']])
print('Pairs a person marked rising:')
display(usable[usable['human_visual_label'] == 'water_level_rising'])

pair_id,coverage_delta_pp,human_visual_label,matched_gauge_delta_ft
A1_A2,5.0,water_level_rising,0.30
A2_A3,7.0,water_level_rising,0.60
A3_A4,6.0,water_level_rising,0.50
B1_B2,-4.0,water_level_falling,-0.40
B2_B3,-5.0,water_level_falling,-0.40
B3_B4,-6.0,water_level_falling,-0.40
C1_C2,0.0,no_water_level_change,0.01
C2_C3,1.0,no_water_level_change,0.00
C3_C4,-1.0,no_water_level_change,-0.01
D1_D2,5.0,water_level_rising,0.40


Pairs a person marked rising:


comparison_status,coverage_change_pp_per_hour,coverage_delta_pp,earlier_id,elapsed_minutes,event_id,gauge_relationship_interpretation,human_label_source,human_review_before_gauge,human_visual_label,is_demo,later_id,matched_gauge_delta_ft,newly_water_pixels,no_longer_water_pixels,pair_id,spatial_change_note,split,training_ready
usable,20.0,5.0,A1,15,A,not_assessed,invented_for_demo_not_computed_from_gauge,True,water_level_rising,True,A2,0.3,NaN,NaN,A1_A2,Cannot derive spatial overlap from totals; real masks required.,train,False
usable,28.0,7.0,A2,15,A,not_assessed,invented_for_demo_not_computed_from_gauge,True,water_level_rising,True,A3,0.6,NaN,NaN,A2_A3,Cannot derive spatial overlap from totals; real masks required.,train,False
usable,24.0,6.0,A3,15,A,not_assessed,invented_for_demo_not_computed_from_gauge,True,water_level_rising,True,A4,0.5,NaN,NaN,A3_A4,Cannot derive spatial overlap from totals; real masks required.,train,False
usable,20.0,5.0,D1,15,D,not_assessed,invented_for_demo_not_computed_from_gauge,True,water_level_rising,True,D2,0.4,NaN,NaN,D1_D2,Cannot derive spatial overlap from totals; real masks required.,test,False
usable,28.0,7.0,D2,15,D,not_assessed,invented_for_demo_not_computed_from_gauge,True,water_level_rising,True,D3,NaN,NaN,NaN,D2_D3,Cannot derive spatial overlap from totals; real masks required.,test,False
usable,24.0,6.0,D3,15,D,not_assessed,invented_for_demo_not_computed_from_gauge,True,water_level_rising,True,D4,NaN,NaN,NaN,D3_D4,Cannot derive spatial overlap from totals; real masks required.,test,False


## 3. Gauge evidence stays optional and separate
D3 has a candidate reading 21 minutes away: it is retained for explanation but is NOT a match.
Never use `candidate_gauge_value` to fill a missing valid match.
Joined values below demonstrate how to inspect a future calibration dataset; they are not evidence of a physical relationship.

In [4]:
paired_with_gauge = usable.dropna(subset=['matched_gauge_delta_ft'])
display(paired_with_gauge[['pair_id','coverage_delta_pp','matched_gauge_delta_ft']])
display(observations[observations['gauge_match_status'] != 'matched'][
    ['observation_id','candidate_gauge_value','gauge_time_gap_minutes','matched_gauge_value']])
print('No calibration is fitted: these values were invented.')

pair_id,coverage_delta_pp,matched_gauge_delta_ft
A1_A2,5.0,0.30
A2_A3,7.0,0.60
A3_A4,6.0,0.50
B1_B2,-4.0,-0.40
B2_B3,-5.0,-0.40
B3_B4,-6.0,-0.40
C1_C2,0.0,0.01
C2_C3,1.0,0.00
C3_C4,-1.0,-0.01
D1_D2,5.0,0.40


observation_id,candidate_gauge_value,gauge_time_gap_minutes,matched_gauge_value
D3,4.2,21,NaN


No calibration is fitted: these values were invented.


## 4. High-and-steady is not automatically normal
The C sequence remains well above the invented 30% reference while barely changing between frames.
No flood status has been established. This separates recent direction from current extent.

In [5]:
display(observations[observations['event_id'] == 'C'][
    ['observation_id','water_fraction_pct','baseline_delta_pp','flood_status']])
display(comparisons[comparisons['event_id'] == 'C'][
    ['pair_id','coverage_delta_pp','human_visual_label']])

observation_id,water_fraction_pct,baseline_delta_pp,flood_status
C1,52.0,22.0,not_established
C2,52.0,22.0,not_established
C3,53.0,23.0,not_established
C4,52.0,22.0,not_established


pair_id,coverage_delta_pp,human_visual_label
C1_C2,0.0,no_water_level_change
C2_C3,1.0,no_water_level_change
C3_C4,-1.0,no_water_level_change


## 5. Inspect failures instead of treating them as zero
E2: glare contaminates the draft; its 61% is rejected. E3: correct-looking mask, shifted camera.
E4: fog, no mask. Newly covered and uncovered areas require real spatial masks;
you cannot reconstruct those areas from two coverage totals alone.

In [6]:
display(observations[observations['event_id'] == 'E'][
    ['observation_id','input_quality','draft_water_pixels','accepted_water_pixels',
     'hypothetical_segmentation_usable','hypothetical_change_usable','note']])
display(comparisons[comparisons['event_id'] == 'E'][
    ['pair_id','comparison_status','coverage_delta_pp','human_visual_label']])

observation_id,input_quality,draft_water_pixels,accepted_water_pixels,hypothetical_segmentation_usable,hypothetical_change_usable,note
E1,clear,44000.0,44000.0,True,True,Illustrative reviewed water mask; image and mask bytes not included.
E2,glare,61000.0,NaN,False,False,Draft includes bright land as water; reject 61% as a usable measurement.
E3,camera_moved,47000.0,47000.0,True,False,"Water mask may be correct for this frame, but old reference is not aligned."
E4,fog,NaN,NaN,False,False,No trustworthy mask; missing does not mean zero water.


pair_id,comparison_status,coverage_delta_pp,human_visual_label
E1_E2,unavailable,NaN,cannot_judge_water_level
E2_E3,unavailable,NaN,cannot_judge_water_level
E3_E4,unavailable,NaN,cannot_judge_water_level


## 6. Query with SQL inside Python
JSONL is ordinary structured data. Pandas, Python dictionaries, and SQLite can all query it.
A real dataset can later use Parquet for efficiency without changing the basic relationships.

In [7]:
import sqlite3
db = sqlite3.connect(':memory:')
observations[['observation_id','event_id','split','water_fraction_pct',
              'matched_gauge_value','hypothetical_change_usable']].to_sql('observations', db, index=False)
display(pd.read_sql_query("SELECT observation_id, water_fraction_pct, matched_gauge_value FROM observations WHERE hypothetical_change_usable = 1 AND water_fraction_pct >= 45", db))
db.close()

observation_id,water_fraction_pct,matched_gauge_value
A4,48.0,4.60
B1,48.0,4.60
C1,52.0,4.90
C2,52.0,4.91
C3,53.0,4.91
C4,52.0,4.90
D4,49.0,4.70


## 7. Check identities, timestamps, and event splits
These checks validate the dummy package, not whether it is sufficient to train a model.
One-site chronological evaluation cannot demonstrate transfer to unseen rivers.
This tiny selection is deliberately illustrative and not balanced or representative enough for a benchmark.

In [8]:
assert len(observations) == 20 and len(comparisons) == 15
assert observations['observation_id'].is_unique
assert observations.groupby('event_id')['split'].nunique().max() == 1
by_id = observations.set_index('observation_id')
for row in comparisons.to_dict('records'):
    a, b = by_id.loc[row['earlier_id']], by_id.loc[row['later_id']]
    assert a['split'] == b['split'] == row['split']
    elapsed = (pd.Timestamp(b['captured_at_utc']) - pd.Timestamp(a['captured_at_utc'])).total_seconds()/60
    assert elapsed == row['elapsed_minutes']
    if row['comparison_status'] == 'usable':
        assert a['view_id'] == b['view_id']
        assert abs((b['water_fraction_pct']-a['water_fraction_pct'])-row['coverage_delta_pp']) < 1e-8
assert observations.loc[observations['observation_id']=='D3','matched_gauge_value'].isna().all()
assert not observations['training_ready'].any()
print('Demo integrity checks passed. No training performed.')
display(observations.groupby(['split','scenario']).size().rename('examples').reset_index())

Demo integrity checks passed. No training performed.


split,scenario,examples
test,Difficult views,4
test,Rising,4
train,Falling,4
train,Rising,4
validation,High and steady,4


## What the real training input would be
For segmentation: source image/crop pixels -> reviewed water-mask pixels. Gauge values are not inputs or targets.
Paths/checksums are null here because there are no assets, not because they are optional in a real training package.
Real records need aligned assets, verified annotation/visibility policy, rights, provenance and a frozen split.
Compare a local pretrained baseline before fine-tuning. Use separate human-reviewed pairs for any later change model.
Historical flood-event labels need independent local evidence; none have been invented as truth in this demo.

Sharing: initially copy this demo folder/ZIP via an external drive. For real data, review permissions,
privacy and licenses before cloud upload. Existing app release export can help after its checks pass;
this proposed multi-table demo does not claim automatic import/export support.